In [1]:
import torch
import random
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader
from tqdm import tqdm
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import GPT2Tokenizer
from transformers import AutoModelForCausalLM
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import nltk
from torch.amp import autocast, GradScaler
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
tokenizer_en = AutoTokenizer.from_pretrained("bert-base-uncased")
tokenizer_frn = AutoTokenizer.from_pretrained("camembert-base")
tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-fr")
print(tokenizer.pad_token_id)
print(tokenizer.eos_token_id)
print(tokenizer.bos_token_id)

print(tokenizer.vocab_size)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

59513
0
None
59514


/usr/local/lib/python3.10/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [3]:
dataset = load_dataset("opus_books", "en-fr")  # Returns a DatasetDict
train_data = dataset["train"]  # Extract the train split
batch_size=64
max_seq_len=120


max_en_length = max(len(ex["translation"]["en"].split()) for ex in train_data)
max_fr_length = max(len(ex["translation"]["fr"].split()) for ex in train_data)


print(f"Longest English sentence: {max_en_length} words")
print(f"Longest French sentence: {max_fr_length} words")

# Shuffle the dataset
train_data = train_data.shuffle(seed=42)

# Define split sizes
train_size = int(0.8 * len(train_data))  # 80% training
val_size = int(0.1 * len(train_data))    # 10% validation
test_size = len(train_data) - train_size - val_size  # 10% test

# Split using .select()
train_dataset = train_data.select(range(train_size))
val_dataset = train_data.select(range(train_size, train_size + val_size))
test_dataset = train_data.select(range(train_size + val_size, len(train_data)))

print(f"Train: {len(train_dataset)}, Validation: {len(val_dataset)}, Test: {len(test_dataset)}")

def collate_fn(batch):
    en_texts = [ex["translation"]["en"] for ex in batch]
    fr_texts = [ex["translation"]["fr"] for ex in batch]

    # Tokenize source (English)
    en_inputs = tokenizer(
        en_texts,
        padding="max_length",  # Changed to fixed-length padding
        truncation=True,
        max_length=max_seq_len,
        return_tensors="pt"
    )

    # Tokenize target (French) with EOS at start and end
    fr_texts = [f"{tokenizer.eos_token} {text} {tokenizer.eos_token}" for text in fr_texts]
    fr_targets = tokenizer(
        fr_texts,
        padding="max_length",
        truncation=True,
        max_length=max_seq_len + 1,  # +1 to account for added EOS
        #add_special_tokens=False,
        return_tensors="pt"
    )

    return {
        "input_ids": en_inputs["input_ids"],
        "attention_mask": en_inputs["attention_mask"],
        "labels": fr_targets["input_ids"]
    }

# Remove these lines from training/eval loops:
# bos_column = torch.full(...)
# labels = torch.cat([bos_column, labels], dim=1)
# Create DataLoaders
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
validation_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)  
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)  

print(len(train_dataloader))

batch = next(iter(train_dataloader))
print(batch["input_ids"].shape)  # (batch_size, max_seq_length)
print(batch["labels"].shape)  # (batch_size, max_seq_length)

print(batch["input_ids"])
print(batch["attention_mask"])
print(batch["labels"])


README.md:   0%|          | 0.00/28.1k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/127085 [00:00<?, ? examples/s]

Longest English sentence: 372 words
Longest French sentence: 324 words
Train: 101668, Validation: 12708, Test: 12709
1589
torch.Size([64, 120])
torch.Size([64, 121])
tensor([[13915,    18,   444,  ..., 59513, 59513, 59513],
        [   58, 30119,     2,  ..., 59513, 59513, 59513],
        [  375,    95,  1308,  ..., 59513, 59513, 59513],
        ...,
        [  233,     4,   281,  ..., 59513, 59513, 59513],
        [ 4958,  1672,    21,  ..., 59513, 59513, 59513],
        [ 4911,   101,  1991,  ..., 59513, 59513, 59513]])
tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])
tensor([[    0,   267,  1624,  ..., 59513, 59513, 59513],
        [    0,   870,   236,  ..., 59513, 59513, 59513],
        [    0,    84,   940,  ..., 59513, 59513, 59513],
        ...,
        [    0,   496,   281,  ..., 59513, 59513, 59513],
        [

In [4]:
class Add_Norm(nn.Module):
    def __init__(self, d_model, dropout):
        super(Add_Norm, self).__init__()
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, new, old):
        return self.norm(old + self.dropout(new))

class VanillaEncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super(VanillaEncoderLayer, self).__init__()
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.addnorm_attn = Add_Norm(d_model, dropout)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, 4*d_model),
            nn.ReLU(),
            nn.Linear(4*d_model, d_model)
        )
        self.addnorm_ffn = Add_Norm(d_model, dropout)

    def forward(self, x, key_padding_mask=None):
        # Use key_padding_mask for padding tokens
        attn_output, _ = self.self_attn(x, x, x, key_padding_mask=key_padding_mask)
        x = self.addnorm_attn(attn_output, x)
        ff_output = self.feed_forward(x)
        x = self.addnorm_ffn(ff_output, x)
        return x

class Encoder(nn.Module):
    def __init__(
        self, 
        d_model, 
        n_heads, 
        dropout, 
        n_layer, 
        vocab_size,
        max_seq_len=512  # Add max sequence length for positional encoding
    ):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        # Add positional encoding (learned example)
        self.pos_embedding = nn.Parameter(torch.randn(1, max_seq_len, d_model))
        self.layers = nn.ModuleList(
            [VanillaEncoderLayer(d_model, n_heads, dropout) for _ in range(n_layer)]
        )
        #self.final_norm = nn.LayerNorm(d_model)  # Optional, depending on design choice

    def forward(self, x, attention_mask=None):
        x = self.embedding(x) + self.pos_embedding[:, :x.size(1), :]  # Add positional encoding
        key_padding_mask = None
        if attention_mask is not None:
            key_padding_mask = (attention_mask == 0) 
        for layer in self.layers:
            x = layer(x, key_padding_mask=key_padding_mask)
        #x = self.final_norm(x)  # Remove if following original Transformer strictly
        return x

In [5]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super(DecoderLayer, self).__init__()
        self.self_attention = nn.MultiheadAttention(d_model, n_heads, dropout=dropout)
        self.encoder_attention = nn.MultiheadAttention(d_model, n_heads, dropout=dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4*d_model),
            nn.ReLU(),
            nn.Linear(4*d_model, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, target, encoder_output, trg_mask=None, src_mask=None):
        # Self-attention
        target_transpose = target.transpose(0, 1)
        _target_self, _ = self.self_attention(
            target_transpose, target_transpose, target_transpose, 
            attn_mask=trg_mask
        )
        _target_self = _target_self.transpose(0, 1)
        target = self.norm1(target + self.dropout1(_target_self))

        # Encoder attention
        target_transpose = target.transpose(0, 1)
        encoder_output_transpose = encoder_output.transpose(0, 1)
        _target_enc, attn_weights = self.encoder_attention(
            target_transpose, encoder_output_transpose, encoder_output_transpose,
            attn_mask=src_mask
        )
        _target_enc = _target_enc.transpose(0, 1)
        target = self.norm2(target + self.dropout2(_target_enc))

        # Feed forward
        _target_ffn = self.ffn(target)
        target = self.norm3(target + self.dropout3(_target_ffn))
        return target


class Decoder(nn.Module):
    def __init__(self,vocab_size,d_model,n_heads,n_layer,dropout,max_len=512):
        super(Decoder,self).__init__()
        self.d_model=d_model
        self.vocab_size=vocab_size
        self.n_heads=n_heads
        self.n_layer=n_layer
        self.dropout=dropout

        self.embedding=nn.Embedding(vocab_size,d_model)
        self.position_encodding=nn.Parameter(torch.randn(1, max_len, d_model))
        self.dropout=nn.Dropout(dropout)

        self.decoder_layers=nn.ModuleList([
            DecoderLayer(
                d_model,
                n_heads,
                dropout
            )
            for _ in range(n_layer)
        ])

        self.logits=nn.Linear(d_model,vocab_size)

    def forward(self,target,encoder_output,trg_mask=False,src_mask=None):
        
        Batch,length=target.shape
        causal_mask=None
        if trg_mask is True:
            causal_mask = torch.triu(torch.ones(length,
                                                length,
                                                device=target.device), diagonal=1).bool()
        scale = torch.sqrt(torch.tensor(self.d_model, dtype=target.dtype, device=target.device))
        target = self.embedding(target) * scale
        target+=self.position_encodding[:,:length,:]
        target=self.dropout(target)

        for layers in self.decoder_layers:
            target=layers(target,encoder_output,causal_mask,src_mask)

        output=self.logits(target)
        return output
        

In [6]:
d_model = 512
encoder_n_heads=8
decoder_n_heads=8
dropout=0.3
vocab_size=tokenizer.vocab_size
encoder_n_layer=4
decoder_n_layer=4
encoder = Encoder(
    d_model=d_model,
    n_heads=encoder_n_heads,
    dropout=dropout,
    n_layer=encoder_n_layer,
    vocab_size=vocab_size
    
).to('cuda')

decoder = Decoder(
    d_model=d_model,
    n_heads=decoder_n_heads,
    vocab_size=vocab_size,
    n_layer=decoder_n_layer,
    dropout=dropout
).to('cuda')

encoder=nn.DataParallel(encoder)
decoder=nn.DataParallel(decoder)


encoder_params = sum(p.numel() for p in encoder.parameters())
decoder_params = sum(p.numel() for p in decoder.parameters())

print(f"Encoder Parameters: {encoder_params:,}")
print(f"Decoder Parameters: {decoder_params:,}")

Encoder Parameters: 43,342,848
Decoder Parameters: 78,080,122


In [7]:
def test_model(encoder, decoder, test_dataloader, tokenizer, max_length=50):
    encoder.eval()
    decoder.eval()
    
    total_bleu_score = 0.0
    total_sentences = 0
    smoothie = SmoothingFunction().method4  # BLEU smoothing function

    test_progress_bar = tqdm(test_dataloader, desc="Testing")

    for step, batch in enumerate(test_progress_bar):
        input_ids = batch["input_ids"].to('cuda')
        labels = batch["labels"].to('cuda')
        attention_mask = batch["attention_mask"].to('cuda')
        batch_size = input_ids.shape[0]

        # Encode input
        with torch.no_grad():
            encoder_output = encoder(input_ids, attention_mask)

        # Start decoding
        bos_token_id = tokenizer.eos_token_id
        generated_tokens = torch.full((batch_size, 1), bos_token_id, dtype=torch.long, device='cuda')

        for _ in range(max_length):
            with torch.no_grad():
                logits = decoder(generated_tokens, encoder_output)
                next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
                generated_tokens = torch.cat([generated_tokens, next_token], dim=1)

                if (next_token == tokenizer.eos_token_id).all():
                    break

        # Decode predictions
        gen_seq = generated_tokens[:, 1:]  # remove BOS token
        pred_sentences = tokenizer.batch_decode(gen_seq, skip_special_tokens=True)
        true_sentences = tokenizer.batch_decode(labels, skip_special_tokens=True)

        # Print first batch
        if step == 0:
            for pred, true in zip(pred_sentences, true_sentences):
                print(f"Pred: {pred}")
                print(f"True: {true}")
                print("-" * 50)

        # Compute BLEU scores
        for pred, true in zip(pred_sentences, true_sentences):
            pred_tokens = pred.strip().split()
            true_tokens = [true.strip().split()]

            hyp_len = len(pred_tokens)
            ref_len = len(true_tokens[0])

            if hyp_len == 0 or ref_len == 0:
                continue

            try:
                bleu_score = sentence_bleu(true_tokens, pred_tokens, smoothing_function=smoothie)
                total_bleu_score += bleu_score
                total_sentences += 1
            except ZeroDivisionError:
                print(f"Skipped BLEU for: Pred='{pred}', True='{true}'")
                continue

        test_progress_bar.set_postfix(
            bleu_score=total_bleu_score / total_sentences if total_sentences > 0 else 0
        )

    avg_bleu_score = total_bleu_score / total_sentences if total_sentences > 0 else 0
    print(f"Final Test BLEU Score: {avg_bleu_score:.4f}")


In [8]:
num_epochs=30
lr=0.0001
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
optimizer = optim.AdamW(
    list(encoder.parameters()) + list(decoder.parameters()),
    lr=lr,
    weight_decay=0.02,       # Adjust weight decay as necessary
    betas=(0.9, 0.999)       # Set beta1 and beta2 values
)
scaler = GradScaler(init_scale=65536.0, growth_factor=2.0, backoff_factor=0.5, growth_interval=2000, enabled=True)

early_stop_threshold = 1.2

for epoch in range(num_epochs):
    encoder.train()
    decoder.train()
    total_train_loss=0
    train_progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    for step, batch in enumerate(train_progress_bar):
        input_ids = batch["input_ids"].to('cuda')
        attention_mask = batch["attention_mask"].to('cuda')
        labels = batch["labels"].to('cuda')
        tgt_input = labels[:, :-1] 
        tgt_output = labels[:, 1:]
        with autocast(device_type='cuda'):
            encoder_output=encoder(input_ids,attention_mask)
            logits=decoder(tgt_input,encoder_output,trg_mask=True)
            loss=criterion(logits.reshape(-1, logits.size(-1)),tgt_output.reshape(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(list(encoder.parameters()) + list(decoder.parameters()), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        total_train_loss += loss.item()
        train_progress_bar.set_postfix({"Train Loss": loss.item()})
    avg_train_loss = total_train_loss / len(train_dataloader)
    print(f"Epoch {epoch + 1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}")

    encoder.eval()
    decoder.eval()
    total_eval_loss = 0

    with torch.no_grad():
        eval_progress_bar = tqdm(validation_dataloader, desc=f"Evaluating Epoch {epoch+1}/{num_epochs}")
        for batch in eval_progress_bar:
            input_ids = batch["input_ids"].to('cuda')
            attention_mask = batch["attention_mask"].to('cuda')
            labels = batch["labels"].to('cuda')
            tgt_input = labels[:, :-1]
            tgt_output = labels[:, 1:]

            with autocast(device_type='cuda'):
                encoder_output = encoder(input_ids,attention_mask)
                logits = decoder(tgt_input, encoder_output,trg_mask=True)
                loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_output.reshape(-1))

            total_eval_loss += loss.item()
            eval_progress_bar.set_postfix({"Eval Loss": loss.item()})
    
    avg_eval_loss = total_eval_loss / len(validation_dataloader)
    print(f"Epoch {epoch + 1}/{num_epochs}, Eval Loss: {avg_eval_loss:.4f}")
    
    if avg_eval_loss > early_stop_threshold * avg_train_loss:
        print(f"Early stopping triggered: Validation loss ({avg_eval_loss:.4f}) is significantly greater than Training loss ({avg_train_loss:.4f}).")
        break

Epoch 1/30: 100%|██████████| 1589/1589 [12:03<00:00,  2.20it/s, Train Loss=3.87]


Epoch 1/30, Train Loss: 4.5529


Evaluating Epoch 1/30: 100%|██████████| 199/199 [00:40<00:00,  4.91it/s, Eval Loss=3.71]


Epoch 1/30, Eval Loss: 3.6985


Epoch 2/30: 100%|██████████| 1589/1589 [12:01<00:00,  2.20it/s, Train Loss=3.33]


...

Epoch 21/30, Eval Loss: 1.9739


Epoch 22/30: 100%|██████████| 1589/1589 [11:59<00:00,  2.21it/s, Train Loss=1.83]


Epoch 22/30, Train Loss: 1.6394


Evaluating Epoch 22/30: 100%|██████████| 199/199 [00:40<00:00,  4.94it/s, Eval Loss=2.08]

Epoch 22/30, Eval Loss: 1.9808
Early stopping triggered: Validation loss (1.9808) is significantly greater than Training loss (1.6394).


In [9]:
test_model(encoder, decoder, test_dataloader, tokenizer)

Testing:   1%|          | 1/199 [00:04<15:14,  4.62s/it, bleu_score=0.168]

Pred: -- Non, grâce, je viens de m'en avaler. -- Non, grâce, grâce, grâce, grâce, je viens de
True: —Non, merci, je sors d'avaler le mien.
--------------------------------------------------
Pred: Un milieu de la nuit, au milieu du cœur, un silence déchirant les écrasant, les écrasant sur la tête de Camille. Un silence, un silence, un
True: Au milieu de la nuit et du silence navré qui traînait, le furieux serrement de mains qu'ils échangeaient était comme un poids écrasant jeté sur la tête de Camille pour le maintenir sous l'eau.
--------------------------------------------------
Pred: Tout se précipita aussitôt et parvint à retenir le mourir le garçon, qui, en sesenta peu à peu à peu à peu à peu à p
True: Tous rentrèrent aussitôt et parvinrent à maintenir l'enfant mourant, qui voulait se jeter hors de son lit, pendant que Gédéon Spilett, lui prenant le bras, sentait son pouls remonter peu à peu...
--------------------------------------------------
Pred: Collins, avait pris lattention d

Testing: 100%|██████████| 199/199 [13:04<00:00,  3.94s/it, bleu_score=0.156]

Final Test BLEU Score: 0.1556
